# 21 — Image Processing

Thin notebook: it only **imports**, **calls** `src/image_processing.py`, and **displays**.
It prepares the pixels of the figures chosen in `20`. Each figure is rotated by its labelled `rotation_deg`, resized so its long side matches the target size, and padded to a square. This runs once for every size in `processing.sizes`.

**Input:** `selection/sets_union.csv`, whose source files are Stage 04's raw copies in `joined/approved_images/`.
**Output:** `<paths.pipeline_root>/processed/<size>/<batch>/<patent>/*.png` plus a `manifest.csv` for each size.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd()
while not (ROOT / 'config.yaml').exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
from src.config_loader import load_config
from src import figure_selection as fs, image_processing as ip

cfg = load_config()
union = pd.read_csv(fs.selection_dir(cfg) / 'sets_union.csv', keep_default_na=False, na_values=[''])
print(len(union), 'figures to process at sizes', cfg['processing']['sizes'])

## Process
Files that already exist are skipped. Set `FORCE = True` after changing `pad_fill` or `sharpen`.

In [ ]:
FORCE = False
manifests = {size: ip.process_all(union, cfg, size, force=FORCE) for size in cfg['processing']['sizes']}

## Checks
Two checks: how many figures had to be **upscaled** because their long side was smaller than the target size, and a visual comparison of some rotated figures before and after.

In [ ]:
for size, man in manifests.items():
    print(f"{size}px: {len(man)} figures, upscaled {int(man['upscaled'].sum())}, "
          f"rotated {int((man['rotation_deg'] != 0).sum())}")
man = manifests[max(manifests)]
man[['orig_w', 'orig_h']].max(axis=1).describe()

In [ ]:
sample = pd.concat([man[man['rotation_deg'] != 0].head(3), man[man['rotation_deg'] == 0].head(3)])
fig, axes = plt.subplots(2, len(sample), figsize=(3 * len(sample), 6))
for i, r in enumerate(sample.itertuples()):
    axes[0, i].imshow(Image.open(r.src)); axes[0, i].set_title(f'raw, rot {r.rotation_deg}', fontsize=8)
    axes[1, i].imshow(Image.open(r.path)); axes[1, i].set_title(f'{r.size}px', fontsize=8)
for ax in axes.ravel():
    ax.axis('off')
plt.tight_layout(); plt.show()